# Benchmark reproducible — Qwen2.5-1.5B-Instruct para formulación de optimización

Este notebook hace cuatro cosas:

1. carga el **benchmark auditado de 30 problemas**;
2. ejecuta **Qwen2.5-1.5B-Instruct base** con direct prompting sobre los 30;
3. ejecuta además el **caso oftalmológico de la Entrega 1** como diagnóstico separado;
4. evalúa cada formulación con una métrica general **FOM-5** y guarda una tabla reproducible.

**Regla experimental:** los `reference_model` y `requirements` se usan únicamente después de generar la respuesta, en el evaluador. Nunca se entregan a Qwen durante generación. Esto evita leakage/hardcoding.

El promedio oficial de baseline se calcula **solo sobre los 30 problemas**. El caso oftalmológico se reporta aparte.


In [ ]:
# Instalar dependencias (Colab)
!pip -q install -U "transformers>=4.45" accelerate bitsandbytes pandas tqdm requests


In [ ]:
import os, re, json, time, math, statistics, base64, gzip, gc
from pathlib import Path
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

SEED = 42
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_DIR = Path("/content/qwen_lp_benchmark")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Benchmark auditado embebido para que el notebook sea autocontenido.
_payload = gzip.decompress(base64.b64decode('...')).decode("utf-8")
BENCHMARK = json.loads(_payload)

print(f"Problemas de benchmark: {len(BENCHMARK)}")
print(pd.Series([x["audit_status"] for x in BENCHMARK]).value_counts())


## Auditoría de los 30 gold models

Se revisó correspondencia entre enunciado y formulación, sentido del objetivo, decisiones, dominios, balances/recursos, lógica de activación, conectividad y condiciones intertemporales.

Resultado: **30/30 formulaciones son utilizables como referencia**. Algunas tienen notas de modelación (Big-M conservador, restricciones redundantes válidas o formulaciones exponenciales) que el evaluador debe tratar como equivalentes y **no exigir literalmente**.


In [ ]:
audit_df = pd.DataFrame([{
    "id": x["id"],
    "title": x["title"],
    "class": x["model_class"],
    "difficulty": x["difficulty"],
    "status": x["audit_status"],
    "note": x["audit_note"],
    "n_requirements": len(x["requirements"]),
} for x in BENCHMARK])

display(audit_df)
audit_df.to_csv(OUTPUT_DIR / "auditoria_gold_30.csv", index=False, encoding="utf-8-sig")


## Métrica FOM-5 (0 a 5)

La métrica deja de ser específica a p-mediana. Cada respuesta se compara con el enunciado, una lista de requisitos semánticos y el gold **solo durante evaluación**.

| Dimensión | Peso | Qué mide |
|---|---:|---|
| Decisiones y dominios | 15% | Variables necesarias, significado, índices y dominios |
| Función objetivo | 20% | sentido, términos y unidades correctas |
| Restricciones y lógica | 35% | reglas esenciales de negocio, balances, capacidades, enlaces, etc. |
| Validez algebraica | 15% | índices consistentes, símbolos bien usados, linealidad/MILP válida |
| Generalización/calidad | 15% | formulación paramétrica, sin hardcoding innecesario, equivalencias válidas |

Cada dimensión se puntúa 0–5.  
**FOM-5 = 0.15·D + 0.20·O + 0.35·R + 0.15·A + 0.15·G.**

El evaluador debe aceptar formulaciones matemáticamente equivalentes. La explicación en prosa no recibe puntos por sí sola.

Además se registran chequeos deterministas de formato/alertas, pero **no sustituyen** la evaluación semántica.


In [ ]:
OPH_PROBLEM = 'En la ciudad de Concepción se requiere construir 2 centros oftalmológicos para atender las consultas médicas de la población. Existen 3 lugares potenciales donde estos centros pueden ser ubicados: Barrio Norte, Los Carrera y Collao.\n\nDebido al tamaño de la zona, la comuna se ha dividido en 5 sectores de demanda: Concepción Centro, Sector Lomas, Costanera, Barrio Universitario y Nonguén. La población de cada sector, expresada en miles de personas, es la siguiente: Concepción Centro tiene 30 mil habitantes, Sector Lomas 15 mil, Costanera 5 mil, Barrio Universitario 2 mil y Nonguén 10 mil habitantes.\n\nLas distancias entre los lugares potenciales para construir los centros y los sectores de demanda se expresan en kilómetros. Desde Barrio Norte, las distancias son de 1,3 km a Concepción Centro, 0,5 km a Sector Lomas, 2,5 km a Costanera, 2,0 km a Barrio Universitario y 3,0 km a Nonguén.\n\nDesde Los Carrera, las distancias son de 0,3 km a Concepción Centro, 2,0 km a Sector Lomas, 1,2 km a Costanera, 1,0 km a Barrio Universitario y 1,9 km a Nonguén.\n\nFinalmente, desde Collao, las distancias son de 1,6 km a Concepción Centro, 2,3 km a Sector Lomas, 2,1 km a Costanera, 1,5 km a Barrio Universitario y 0,4 km a Nonguén.\n\nSe debe formular un modelo matemático que permita determinar cuáles 2 de los 3 lugares potenciales deben seleccionarse para instalar los centros oftalmológicos y cómo asignar los sectores de demanda a estos centros, buscando minimizar la distancia de atención considerando la población de cada sector.\n\nPara la formulación del modelo, se deben definir los conjuntos, parámetros y variables de decisión, plantear la función objetivo correspondiente e incorporar las restricciones necesarias para garantizar que cada sector sea atendido por un único centro, que los sectores solo puedan ser asignados a centros que hayan sido construidos y que se instalen exactamente 2 centros oftalmológicos.'
OPH_REFERENCE = 'Sean \\(I\\) los lugares candidatos y \\(J\\) los sectores de demanda. \\(p_j\\) es la población del sector \\(j\\), \\(d_{ij}\\) la distancia desde el lugar \\(i\\) al sector \\(j\\), y \\(K=2\\). \\(y_i\\in\\{0,1\\}\\) vale 1 si se instala un centro en \\(i\\), y \\(x_{ij}\\in\\{0,1\\}\\) vale 1 si el sector \\(j\\) se asigna al centro \\(i\\).\n\n\\[\n\\min \\sum_{i\\in I}\\sum_{j\\in J} p_j d_{ij}x_{ij}\n\\]\n\nsujeto a\n\\[\n\\sum_{i\\in I}x_{ij}=1\\quad \\forall j\\in J,\n\\]\n\\[\nx_{ij}\\le y_i\\quad \\forall i\\in I,\\forall j\\in J,\n\\]\n\\[\n\\sum_{i\\in I}y_i=K,\n\\]\n\\[\nx_{ij},y_i\\in\\{0,1\\}.\n\\]'
OPH_REQUIREMENTS = ['Minimizar distancia total ponderada por la población de cada sector.', 'Abrir exactamente dos lugares candidatos.', 'Asignar cada sector exactamente a un centro.', 'Permitir asignación solo a centros abiertos.', 'Variables binarias separadas para apertura y asignación.', 'Índices y conjuntos deben recorrer los 3 candidatos y los 5 sectores sin hardcoding de sumatorias.']

# Salida histórica guardada en el notebook original de la Entrega 1.
# Se evalúa con la NUEVA métrica solo como referencia diagnóstica.
HISTORICAL_OPH_RESPONSE = '### Modelo Matemático\n\n#### Conjuntos Definidos:\n- **L**: Conjunto de los 3 lugares potenciales (Barrio Norte, Los Carreras, Collao).\n- **S**: Conjunto de los 5 sectores de demanda (Concepción Centro, Sector Lomas, Costanera, Barrio Universitario, Nonguén).\n\n#### Parámetros:\n- \\(d_{ij}\\): Distancia entre el lugar \\(i\\) y el sector \\(j\\), donde \\(i \\in L\\) y \\(j \\in S\\).\n- \\(p_j\\): Población del sector \\(j\\), donde \\(j \\in S\\).\n\n#### Variables de Decisión:\n- \\(x_{ij} = 1\\) si el sector j está asignado al lugar i, 0 en caso contrario.\n\n#### Función Objetivo:\nMinimizar la suma total de las distancias de atención:\n\\[\n\\min \\sum_{i \\in L} \\sum_{j \\in S} d_{ij} x_{ij}\n\\]\n\n#### Restricciones:\n1. Cantidad máxima de centros: Solo se instalan exactamente 2 centros.\n\\[\n\\sum_{i \\in L} x_{ij} = 2 \\quad \\forall j \\in S\n\\]\n\n2. Asignación única de sectores:\n\\[\n\\sum_{i \\in L} x_{ij} = 1 \\quad \\forall j \\in S\n\\]\n\n3. Población mínima requerida:\n\\[\np_j - \\sum_{i \\in L} x_{ij} p_j \\geq 0 \\quad \\forall j \\in S\n\\]\n\n4. Posible construcción de centros: Las distancias deben ser positivas.\n\\[\nd_{ij} > 0 \\quad \\forall i \\in L, j \\in S\n\\]\n\nLuego enumera los datos y propone informalmente una posible asignación.'
HISTORICAL_OLD_SCORE = 1.65

DIAGNOSTIC = {
    "id": "diagnostic_oftalmo",
    "title": "Centros oftalmológicos — Entrega 1",
    "problem": OPH_PROBLEM,
    "reference_model": OPH_REFERENCE,
    "requirements": OPH_REQUIREMENTS,
    "model_class": "MILP/localización",
    "difficulty": "media",
    "audit_status": "Gold construido para diagnóstico",
    "audit_note": "Se excluye del promedio del benchmark.",
}


In [ ]:
# Cargar Qwen base en FP16: un modelo 1.5B cabe en una T4 sin QLoRA.
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

torch.manual_seed(SEED)
np.random.seed(SEED)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)
model.eval()
print("Modelo base cargado:", MODEL_ID)


In [ ]:
def generate_direct(problem: str, max_new_tokens: int = 1200):
    """Baseline limpio: mismo esquema de Direct Prompting, pero decodificación determinista."""
    messages = [{"role": "user", "content": problem}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    t0 = time.perf_counter()
    with torch.inference_mode():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    latency = time.perf_counter() - t0

    generated = out[0][inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(generated, skip_special_tokens=True).strip()
    return {
        "candidate": text,
        "prompt_tokens": int(inputs["input_ids"].shape[1]),
        "output_tokens": int(generated.shape[0]),
        "latency_sec": latency,
    }

def deterministic_diagnostics(text: str):
    s = text.lower()
    return {
        "has_objective": bool(re.search(r"\\min|\\max|\bminimizar\b|\bmaximizar\b", s)),
        "has_constraints": ("restric" in s) or ("s.a." in s) or ("subject to" in s),
        "has_domains": bool(re.search(r"\\in\s*\\?\{|binari|entera|no negativa|>=\s*0|\\geq\s*0", s)),
        "has_sets_or_indices": ("conjunto" in s) or bool(re.search(r"\\forall|\bin\s+[a-z]", s)),
        "has_variables": ("variable" in s) or bool(re.search(r"[xyz]_[{]?", s)),
        "warning_neq": ("\\neq" in s) or ("!=" in s),
        "warning_numeric_sum_bounds": bool(re.search(r"\\sum_\{[a-zA-Z]\s*=\s*\d+", text)),
        "empty_output": len(text.strip()) == 0,
    }


In [ ]:
# Ejecutar los 30 casos + diagnóstico oftalmológico determinista.
# El caso histórico NO se vuelve a generar: se agrega después como comparación.
records = []

for item in tqdm(BENCHMARK + [DIAGNOSTIC], desc="Generando"):
    g = generate_direct(item["problem"])
    diag = deterministic_diagnostics(g["candidate"])
    records.append({
        "id": item["id"],
        "title": item["title"],
        "split": "benchmark" if item["id"].startswith("opt_") else "diagnostic",
        "variant": "baseline_direct_deterministic",
        "model": MODEL_ID,
        "candidate": g["candidate"],
        "prompt_tokens": g["prompt_tokens"],
        "output_tokens": g["output_tokens"],
        "latency_sec": g["latency_sec"],
        **diag,
    })

# Añadir la respuesta histórica de Entrega 1 para poder compararla bajo la misma métrica.
records.append({
    "id": "diagnostic_oftalmo",
    "title": "Centros oftalmológicos — Entrega 1",
    "split": "diagnostic",
    "variant": "entrega1_historical_sampled",
    "model": MODEL_ID,
    "candidate": HISTORICAL_OPH_RESPONSE,
    "prompt_tokens": np.nan,
    "output_tokens": np.nan,
    "latency_sec": np.nan,
    **deterministic_diagnostics(HISTORICAL_OPH_RESPONSE),
})

raw_df = pd.DataFrame(records)
raw_df.to_json(OUTPUT_DIR / "generaciones_raw.jsonl", orient="records", lines=True, force_ascii=False)
raw_df.to_csv(OUTPUT_DIR / "generaciones_raw.csv", index=False, encoding="utf-8-sig")

print("Generaciones:", len(raw_df))
display(raw_df[["id","variant","prompt_tokens","output_tokens","latency_sec","warning_neq","warning_numeric_sum_bounds"]])


## Evaluador semántico

Para que el score sea automático necesitamos un **judge fijo** más fuerte que el modelo de 1.5B. Hay dos opciones:

- `local`: Qwen2.5-7B-Instruct en 4-bit sobre la misma T4. No requiere API.
- `openrouter`: usa un modelo externo fijo. Es más cómodo para repetir muchas variantes.

El judge ve el gold **solo después** de generar. No puede utilizarse como verificador en Best-of-N ni en Repair, porque eso filtraría la respuesta correcta. Las futuras mejoras usarán un `blind_verifier` que verá solo enunciado + candidato.


In [ ]:
# CONFIGURACIÓN DEL JUDGE
JUDGE_MODE = "local"   # "local" o "openrouter"
LOCAL_JUDGE_ID = "Qwen/Qwen2.5-7B-Instruct"

# Si eliges OpenRouter:
OPENROUTER_MODEL = ""  # escribe aquí un modelo fijo disponible en tu cuenta


In [ ]:
RUBRIC_TEXT = r'''
Evalúa una formulación matemática de optimización. Acepta notaciones y formulaciones matemáticamente equivalentes; NO exijas copiar el gold.

Puntúa cada dimensión de 0 a 5:
- decision_domains (15%): variables necesarias, significado, índices y dominios.
- objective (20%): sentido y expresión del objetivo.
- constraints_logic (35%): restricciones esenciales y fidelidad a las reglas del enunciado.
- algebra_validity (15%): coherencia de índices/símbolos y formulación LP/MILP matemáticamente válida.
- generalization (15%): parametrización y ausencia de hardcoding innecesario.

Escala:
0 = ausente/irrelevante
1 = errores fundamentales
2 = parcial con errores graves
3 = mayormente correcta pero con una omisión material
4 = correcta con un problema menor
5 = completamente correcta o matemáticamente equivalente

No premies explicaciones largas. No penalices una formulación equivalente por usar otros nombres de variables, otro Big-M válido, cortes, flujo, MTZ u otra formulación correcta.

Devuelve SOLO JSON válido:
{
  "decision_domains": 0,
  "objective": 0,
  "constraints_logic": 0,
  "algebra_validity": 0,
  "generalization": 0,
  "requirement_results": [
    {"requirement": "...", "status": "met|partial|missing|wrong", "reason": "..."}
  ],
  "critical_errors": ["..."],
  "summary": "..."
}
'''

def make_judge_prompt(item, candidate):
    reqs = "\n".join(f"- {r}" for r in item["requirements"])
    return f'''{RUBRIC_TEXT}

ENUNCIADO:
{item["problem"]}

REQUISITOS SEMÁNTICOS AUDITADOS:
{reqs}

FORMULACIÓN DE REFERENCIA (solo para evaluación):
{item["reference_model"]}

RESPUESTA DEL MODELO A EVALUAR:
{candidate}
'''

def extract_json_object(text):
    text = text.strip()
    # Quitar fences si aparecen
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.I)
    text = re.sub(r"\s*```$", "", text)
    start, end = text.find("{"), text.rfind("}")
    if start < 0 or end < start:
        raise ValueError("El judge no devolvió un objeto JSON.")
    return json.loads(text[start:end+1])

def compute_fom5(j):
    vals = {
        "decision_domains": float(j["decision_domains"]),
        "objective": float(j["objective"]),
        "constraints_logic": float(j["constraints_logic"]),
        "algebra_validity": float(j["algebra_validity"]),
        "generalization": float(j["generalization"]),
    }
    for k,v in vals.items():
        if not 0 <= v <= 5:
            raise ValueError(f"Score fuera de rango: {k}={v}")
    return (
        0.15*vals["decision_domains"] +
        0.20*vals["objective"] +
        0.35*vals["constraints_logic"] +
        0.15*vals["algebra_validity"] +
        0.15*vals["generalization"]
    )

def requirement_coverage(j):
    rr = j.get("requirement_results", [])
    if not rr:
        return np.nan
    weights = {"met":1.0, "partial":0.5, "missing":0.0, "wrong":0.0}
    vals = [weights.get(str(x.get("status","")).lower(), 0.0) for x in rr]
    return float(np.mean(vals))


In [ ]:
# Preparar lookup de gold/reference.
gold_lookup = {x["id"]: x for x in BENCHMARK}
gold_lookup["diagnostic_oftalmo"] = DIAGNOSTIC

if JUDGE_MODE == "local":
    # Liberar el Qwen base para cargar un judge 7B cuantizado.
    del model
    gc.collect()
    torch.cuda.empty_cache()

    from transformers import BitsAndBytesConfig
    qconf = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    judge_tokenizer = AutoTokenizer.from_pretrained(LOCAL_JUDGE_ID)
    if judge_tokenizer.pad_token is None:
        judge_tokenizer.pad_token = judge_tokenizer.eos_token
    judge_model = AutoModelForCausalLM.from_pretrained(
        LOCAL_JUDGE_ID,
        quantization_config=qconf,
        device_map="auto",
    )
    judge_model.eval()
    print("Judge local cargado:", LOCAL_JUDGE_ID)

    def call_judge(prompt):
        messages = [
            {"role":"system","content":"Eres un evaluador estricto de modelos de Investigación de Operaciones."},
            {"role":"user","content":prompt},
        ]
        txt = judge_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inp = judge_tokenizer(txt, return_tensors="pt").to(judge_model.device)
        with torch.inference_mode():
            out = judge_model.generate(
                **inp,
                max_new_tokens=1000,
                do_sample=False,
                pad_token_id=judge_tokenizer.eos_token_id,
            )
        gen = out[0][inp["input_ids"].shape[1]:]
        return judge_tokenizer.decode(gen, skip_special_tokens=True)

elif JUDGE_MODE == "openrouter":
    import requests, getpass
    key = os.environ.get("OPENROUTER_API_KEY") or getpass.getpass("OPENROUTER_API_KEY: ")
    if not OPENROUTER_MODEL.strip():
        raise ValueError("Define OPENROUTER_MODEL con un modelo fijo.")
    def call_judge(prompt):
        r = requests.post(
            "https://openrouter.ai/api/v1/chat/completions",
            headers={"Authorization": f"Bearer {key}", "Content-Type":"application/json"},
            json={
                "model": OPENROUTER_MODEL,
                "temperature": 0,
                "messages":[
                    {"role":"system","content":"Eres un evaluador estricto de modelos de Investigación de Operaciones."},
                    {"role":"user","content":prompt},
                ]
            },
            timeout=180,
        )
        r.raise_for_status()
        return r.json()["choices"][0]["message"]["content"]
else:
    raise ValueError("JUDGE_MODE debe ser 'local' u 'openrouter'.")


In [ ]:
# Evaluar todas las generaciones.
eval_rows = []

for _, rec in tqdm(raw_df.iterrows(), total=len(raw_df), desc="Evaluando FOM-5"):
    item = gold_lookup[rec["id"]]
    prompt = make_judge_prompt(item, rec["candidate"])

    parsed = None
    raw_judge = None
    err = None
    for attempt in range(2):
        try:
            raw_judge = call_judge(prompt)
            parsed = extract_json_object(raw_judge)
            score = compute_fom5(parsed)
            break
        except Exception as e:
            err = str(e)
            parsed = None

    if parsed is None:
        eval_rows.append({
            **rec.to_dict(),
            "score_fom5": np.nan,
            "judge_error": err,
            "judge_raw": raw_judge,
        })
        continue

    eval_rows.append({
        **rec.to_dict(),
        "score_fom5": score,
        "decision_domains": parsed["decision_domains"],
        "objective": parsed["objective"],
        "constraints_logic": parsed["constraints_logic"],
        "algebra_validity": parsed["algebra_validity"],
        "generalization": parsed["generalization"],
        "requirement_coverage": requirement_coverage(parsed),
        "critical_errors": json.dumps(parsed.get("critical_errors",[]), ensure_ascii=False),
        "judge_summary": parsed.get("summary",""),
        "judge_error": None,
        "judge_raw": json.dumps(parsed, ensure_ascii=False),
    })

results_df = pd.DataFrame(eval_rows)
results_df.to_csv(OUTPUT_DIR / "resultados_fom5.csv", index=False, encoding="utf-8-sig")
results_df.to_json(OUTPUT_DIR / "resultados_fom5.jsonl", orient="records", lines=True, force_ascii=False)

cols = ["id","title","variant","score_fom5","decision_domains","objective","constraints_logic",
        "algebra_validity","generalization","requirement_coverage","critical_errors"]
display(results_df[cols])


In [ ]:
# Resumen oficial del baseline = SOLO 30 problemas.
baseline = results_df[
    (results_df["split"] == "benchmark") &
    (results_df["variant"] == "baseline_direct_deterministic")
].copy()

summary = {
    "n": len(baseline),
    "mean_fom5": baseline["score_fom5"].mean(),
    "median_fom5": baseline["score_fom5"].median(),
    "std_fom5": baseline["score_fom5"].std(ddof=1),
    "min_fom5": baseline["score_fom5"].min(),
    "max_fom5": baseline["score_fom5"].max(),
    "mean_requirement_coverage": baseline["requirement_coverage"].mean(),
}
print(json.dumps(summary, indent=2, ensure_ascii=False))

print("\nPromedios por dimensión:")
display(baseline[["decision_domains","objective","constraints_logic","algebra_validity","generalization"]].mean().to_frame("mean"))

print("\nPor dificultad:")
display(baseline.groupby("id", as_index=False).head(0) if False else
        baseline.merge(pd.DataFrame([{"id":x["id"],"difficulty":x["difficulty"],"model_class":x["model_class"]} for x in BENCHMARK]), on="id")
        .groupby("difficulty")["score_fom5"].agg(["count","mean","median"]).sort_index())

# Diagnóstico oftalmológico: histórico vs nueva corrida determinista.
print("\nCaso oftalmológico (NO entra al promedio):")
display(results_df[results_df["id"]=="diagnostic_oftalmo"][
    ["variant","score_fom5","decision_domains","objective","constraints_logic","algebra_validity","generalization","critical_errors"]
])

print(f"\nNota histórica reportada con la rúbrica antigua: {HISTORICAL_OLD_SCORE}/5.0")
print("No debe compararse numéricamente de forma directa con FOM-5; la salida histórica sí fue re-evaluada con FOM-5 arriba.")


## Contrato para las siguientes mejoras

Las siguientes variantes deben producir exactamente el mismo esquema de resultados (`id`, `variant`, `candidate`) y después pasar por **el mismo evaluador FOM-5**:

1. `ir_compiler` — Qwen produce representación intermedia estructurada y un compilador determinista genera la formulación.
2. `best_of_n_blind` — Qwen produce 4–8 candidatos y un verificador **sin gold** escoge uno.
3. `generate_validate_repair` — un verificador **sin gold** devuelve errores específicos y Qwen repara, máximo 1–2 rondas.

**Prohibición metodológica:** el `reference_model`, los `requirements` auditados y el judge FOM-5 jamás pueden usarse para escoger, corregir o regenerar una respuesta. Solo se usan al final para medir. Así evitamos hardcoding/leakage.


In [ ]:
# Función auxiliar para futuras variantes.
# Una futura estrategia solo debe devolver un DataFrame con id/title/candidate y luego reutilizar el evaluador.
def make_variant_record(item, variant_name, candidate, **extra):
    return {
        "id": item["id"],
        "title": item["title"],
        "split": "benchmark" if item["id"].startswith("opt_") else "diagnostic",
        "variant": variant_name,
        "model": MODEL_ID,
        "candidate": candidate,
        **deterministic_diagnostics(candidate),
        **extra,
    }

print("Pipeline listo para incorporar IR+compiler, Best-of-N+blind verifier y Validate→Repair sin reentrenar.")
